<h1>Librerias Necesarias<h1>

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

<h1>1.Cargar y explorar el dataset<h1>

In [ ]:
# Configuración de rutas (ajusta según tu estructura de archivos)
dataset_path = 'Urdu-Audio-Digits-Dataset'  # Cambia esto a tu ruta

# Cargar los archivos de audio
def load_audio_files(dataset_path):
    audio_files = []
    labels = []
    
    for digit in os.listdir(dataset_path):
        digit_path = os.path.join(dataset_path, digit)
        if os.path.isdir(digit_path):
            for audio_file in os.listdir(digit_path):
                if audio_file.endswith('.wav'):
                    file_path = os.path.join(digit_path, audio_file)
                    audio_files.append(file_path)
                    labels.append(digit)
    
    return audio_files, labels

audio_files, labels = load_audio_files(dataset_path)

print(f"Total de archivos de audio cargados: {len(audio_files)}")
print(f"Distribución de etiquetas: {np.unique(labels, return_counts=True)}")

<h1>2.Extracción de características con Librosa<h1>

In [ ]:
def extract_features(audio_files, n_mfcc=13, max_pad_len=100):
    features = []
    
    for file in audio_files:
        # Cargar el archivo de audio
        audio, sr = librosa.load(file, sr=None)
        
        # Extraer características MFCC
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        
        # Asegurar longitud consistente (padding o truncamiento)
        pad_width = max_pad_len - mfccs.shape[1]
        if pad_width > 0:
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]
        
        features.append(mfccs)
    
    return np.array(features)

# Extraer características (puede tomar tiempo)
features = extract_features(audio_files)

print(f"Forma de las características extraídas: {features.shape}")

<h1>3.Visualización de características<h1>

In [ ]:
# Visualizar MFCCs de un ejemplo
def plot_mfcc(mfccs, sr=22050):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mfccs, x_axis='time', sr=sr)
    plt.colorbar()
    plt.title('MFCC')
    plt.tight_layout()
    plt.show()

# Mostrar el primer ejemplo
plot_mfcc(features[0])

<h1>4.Preprocesamiento de datos<h1>

In [ ]:
# Codificar etiquetas
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
one_hot_labels = to_categorical(encoded_labels)

# Reformar características para el MLP
# (aplanamos cada matriz MFCC en un vector 1D)
X = features.reshape(features.shape[0], -1)
y = one_hot_labels

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")

<h1>5.Construcción del modelo MLP<h1>

In [ ]:
def build_mlp_model(input_shape, num_classes):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_shape,)),
        Dropout(0.3),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

num_classes = len(label_encoder.classes_)
input_shape = X_train.shape[1]

model = build_mlp_model(input_shape, num_classes)
model.summary()

<h1>6.Entrenamiento del modelo<h1>

In [ ]:
# Callback para detener el entrenamiento si no hay mejora
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stopping],
                    verbose=1)

<h1>7.Evaluación del modelo<h1>

In [ ]:
# Evaluar en el conjunto de prueba
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nPrecisión en el conjunto de prueba: {test_acc:.4f}")

# Graficar precisión y pérdida durante el entrenamiento
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Precisión durante el entrenamiento')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Pérdida durante el entrenamiento')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()

plt.tight_layout()
plt.show()

<h1>8.Predicción en nuevos ejemplos<h1>

In [ ]:
def predict_digit(audio_path, model, label_encoder):
    # Extraer características
    features = extract_features([audio_path])
    features = features.reshape(1, -1)
    
    # Predecir
    pred = model.predict(features)
    pred_class = np.argmax(pred, axis=1)
    pred_label = label_encoder.inverse_transform(pred_class)[0]
    confidence = np.max(pred)
    
    return pred_label, confidence

# Ejemplo de predicción (reemplaza con la ruta a un archivo de audio)
# audio_path = 'path_to_your_audio_file.wav'
# pred_label, confidence = predict_digit(audio_path, model, label_encoder)
# print(f"Predicción: Dígito {pred_label} con confianza {confidence:.2f}")